# Phase 2: Deterministic Temperature Sensor Monitoring

This notebook ingests `artifacts/aligned_phase1_temperature.csv`, derives the run-specific 90th percentile from `artifacts/phase1_average_statistics.csv`, applies the deterministic re-sync state machine, and exports `artifacts/phase2_temperature_sensors.csv`.

Related documentation to consult while working through this notebook:
- `Distributed_Monitoring_POC_Project_Plan.md` for the canonical design and policy rules.
- `scripts/README.md` for the Phase 1/Phase 2 workflow overview.
- `instructions/PYTHON_ENV_SETUP_GUIDE.md` for environment setup details.
- `instructions/distributed_monitoring_notebook_required_changes.md` for the implementation notes that shaped the current logic.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

## Stage 1: Input Loading and Policy Resolution

This stage validates input artifacts, resolves preferred and fallback source policies, and normalizes numeric columns required by the deterministic state machine.

Run context note: repository paths resolve from `Path.cwd()` first and fall back to its parent when the notebook is executed from the `scripts` directory. If artifacts still are not found under the resolved project root, run the notebook from the expected repository root or adjust path resolution.

In [ ]:
# Stage 1: Load and normalize Phase 1 inputs for deterministic processing.
def resolve_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "artifacts").exists() and (candidate / "scripts").exists():
            return candidate
    return Path.cwd()

REPO_ROOT = resolve_repo_root()
PHASE1_ALIGNED_PATH = REPO_ROOT / "artifacts/aligned_phase1_temperature.csv"
PHASE1_STATS_PATH = REPO_ROOT / "artifacts/phase1_average_statistics.csv"
PHASE2_OUTPUT_PATH = REPO_ROOT / "artifacts/phase2_temperature_sensors.csv"
PHASE2_METRICS_PATH = REPO_ROOT / "artifacts/phase2_temperature_metrics.json"

# Strict all-devices policy: use the all-devices p90 threshold and derive the row-level
# observed global average from the current minute's sensor readings.
PRIMARY_SOURCE_SERIES_NAME = "all_devices_including_imputed"
PRIMARY_GLOBAL_AVG_COLUMN = "average_temperature_all_devices"

SOURCE_SERIES_NAME = PRIMARY_SOURCE_SERIES_NAME
GLOBAL_AVG_COLUMN = PRIMARY_GLOBAL_AVG_COLUMN

def require_input_files(paths):
    for path in paths:
        if not path.exists():
            raise FileNotFoundError(f"Required input not found: {path}")


def resolve_global_average_column(df):
    # Use the all-device aggregate as the strict global source.
    if PRIMARY_GLOBAL_AVG_COLUMN in df.columns:
        return PRIMARY_GLOBAL_AVG_COLUMN
    raise ValueError(
        "Could not find the required all-devices global average column in aligned phase1 data"
    )


def select_p90_row(stats_frame):
    # Use the all-devices p90 row as the strict threshold source.
    row = stats_frame.loc[stats_frame["series_name"] == PRIMARY_SOURCE_SERIES_NAME]
    if row.empty:
        raise ValueError(
            "Could not find the required all-devices p90 source row in phase1_average_statistics.csv"
        )
    return row


def find_sensor_columns(df):
    # Per-sensor columns follow *_temperature; exclude aggregate averages.
    cols = sorted(
        [
            c
            for c in df.columns
            if c.endswith("_temperature")
            and c not in [PRIMARY_GLOBAL_AVG_COLUMN]
        ]
    )
    if not cols:
        raise ValueError("No per-sensor temperature columns found in aligned phase1 data")
    return cols


require_input_files([PHASE1_ALIGNED_PATH, PHASE1_STATS_PATH])

phase1_df = pd.read_csv(PHASE1_ALIGNED_PATH)
stats_df = pd.read_csv(PHASE1_STATS_PATH)

if phase1_df.empty:
    raise ValueError("aligned_phase1_temperature.csv has no rows")

GLOBAL_AVG_COLUMN = resolve_global_average_column(phase1_df)
SOURCE_SERIES_NAME = PRIMARY_SOURCE_SERIES_NAME
p90_row = select_p90_row(stats_df)

p90_threshold = float(p90_row.iloc[0]["p90"])
sensor_cols = find_sensor_columns(phase1_df)

phase1_df = phase1_df.sort_values("bucket_epoch", kind="mergesort").reset_index(drop=True)
phase1_df[GLOBAL_AVG_COLUMN] = pd.to_numeric(phase1_df[GLOBAL_AVG_COLUMN], errors="coerce")
for col in sensor_cols:
    phase1_df[col] = pd.to_numeric(phase1_df[col], errors="coerce")

first_xbar = float(phase1_df.loc[0, GLOBAL_AVG_COLUMN])
if np.isnan(first_xbar):
    non_null_global = phase1_df[GLOBAL_AVG_COLUMN].dropna()
    if non_null_global.empty:
        raise ValueError("Global average column has no numeric values")
    first_xbar = float(non_null_global.iloc[0])

print(
    json.dumps(
        {
            "repo_root": str(REPO_ROOT),
            "rows": int(len(phase1_df)),
            "sensor_count": int(len(sensor_cols)),
            "p90_threshold": p90_threshold,
            "global_avg_column": GLOBAL_AVG_COLUMN,
            "source_series_name": SOURCE_SERIES_NAME,
        },
        indent=2,
    )
)

## Stage 2: Deterministic Re-sync State Machine

This stage applies row-by-row state transitions, computes local trigger requests, tracks re-sync causes, and writes the detailed Phase 2 operational dataset.

State transition contract per row:
- Read the `entry_*` state carried into this minute.
- Read the `observed_*` values for this minute bucket.
- Compute `event_*` outcomes using only `entry_*` references and margins for local trigger evaluation.
- Commit the resulting `exit_*` state at the end of the iteration; that state becomes the next row's `entry_*` state.

Row invariants:
- All `entry_*` fields are the state used to evaluate this row.
- All `observed_*` fields are raw values from the current minute bucket.
- All `event_*` fields are outcomes produced during this minute.
- All `exit_*` fields describe the state that leaves this row and becomes active next row.

Transition summary note:
- `transition_summary` is a compact pipe-delimited summary of the row's event and exit-state changes, placed early in the output for audit readability.

In [ ]:
def round_or_blank(value):
    return "" if np.isnan(value) else round(float(value), 4)


def compute_resync_update(row, sensor_columns, global_average, fallback_xbar, threshold):
    # Re-sync recomputes a shared margin and refreshes available sensor references.
    if np.isnan(global_average):
        computed_xbar = float(fallback_xbar)
    else:
        computed_xbar = float(global_average)
    computed_delta = float(threshold - computed_xbar)

    refreshed_refs = {}
    for sensor_column in sensor_columns:
        sensor_value = row[sensor_column]
        if not np.isnan(sensor_value):
            refreshed_refs[sensor_column] = float(sensor_value)

    refreshed_margins = {sensor_column: computed_delta for sensor_column in sensor_columns}
    return computed_xbar, computed_delta, refreshed_refs, refreshed_margins


def build_transition_summary(
    event_resync_consumed_from_prior_row,
    event_any_sensor_requested_resync,
    event_resync_triggered_by_local_violation,
    event_exit_delta_negative,
    exit_resync_scheduled_next_row,
    event_resync_reason,
):
    parts = []
    if event_resync_consumed_from_prior_row == 1:
        parts.append("consumed_prior_resync")
    if event_any_sensor_requested_resync == 1:
        parts.append("local_request")
    if event_resync_triggered_by_local_violation == 1:
        parts.append("local_resync")
    if event_resync_reason == "sensor_recovered_after_offline":
        parts.append("recovered_after_offline")
    if event_exit_delta_negative == 1:
        parts.append("exit_delta_negative")
    if exit_resync_scheduled_next_row == 1:
        parts.append("scheduled_next_resync")
    if not parts:
        return "steady_state"
    return "|".join(parts)


records = []
sensor_count = len(sensor_cols)

# State at row t: baseline global average, baseline delta, and per-sensor reference/margin.
prior_xbar = first_xbar
prior_delta_global = p90_threshold - prior_xbar
sensor_reference_values = {
    sensor_col: (
        float(phase1_df.loc[0, sensor_col])
        if not np.isnan(phase1_df.loc[0, sensor_col])
        else np.nan
    )
    for sensor_col in sensor_cols
}
local_margin_values = {sensor_col: float(prior_delta_global) for sensor_col in sensor_cols}
sensor_last_resync_included = {
    sensor_col: not np.isnan(phase1_df.loc[0, sensor_col]) for sensor_col in sensor_cols
}
sensor_missing_gap_count = {sensor_col: 0 for sensor_col in sensor_cols}

# This flag is set by row t and consumed by row t+1.
resync_due_next_row = 0

for _, row in phase1_df.iterrows():
    current_sensor_values = [row[sensor_col] for sensor_col in sensor_cols if not np.isnan(row[sensor_col])]
    global_avg = (
        float(np.mean(current_sensor_values))
        if current_sensor_values
        else np.nan
    )

    entry_xbar_t0 = float(prior_xbar)
    entry_delta_global = float(prior_delta_global)

    event_resync_consumed_from_prior_row = int(resync_due_next_row == 1)
    exit_resync_scheduled_next_row = 0
    event_resync_performed = 0
    event_resync_triggered_by_local_violation = 0
    event_resync_reason = ""

    trigger_message_count = 0
    request_message_count = 0
    response_message_count = 0
    broadcast_message_count = 0

    sensor_fields = {}
    triggering_sensor_names = []

    next_prior_xbar = float(prior_xbar)
    next_prior_delta_global = float(prior_delta_global)
    next_sensor_reference_values = dict(sensor_reference_values)
    next_local_margin_values = dict(local_margin_values)

    recovering_sensors = []
    for sensor_col in sensor_cols:
        sensor_val = row[sensor_col]
        if np.isnan(sensor_val):
            sensor_missing_gap_count[sensor_col] += 1
        else:
            if (
                sensor_missing_gap_count[sensor_col] > 0
                and sensor_last_resync_included.get(sensor_col, True) is False
            ):
                recovering_sensors.append(sensor_col)
            sensor_missing_gap_count[sensor_col] = 0

    recovery_resync_row = bool(recovering_sensors)

    forced_resync_row = event_resync_consumed_from_prior_row == 1 or recovery_resync_row
    if forced_resync_row:
        if event_resync_consumed_from_prior_row == 1:
            event_resync_reason = "negative_delta_from_prior_row"
        elif recovery_resync_row:
            event_resync_reason = "sensor_recovered_after_offline"

        (
            computed_xbar,
            computed_delta,
            refreshed_refs,
            refreshed_margins,
        ) = compute_resync_update(
            row=row,
            sensor_columns=sensor_cols,
            global_average=global_avg,
            fallback_xbar=prior_xbar,
            threshold=p90_threshold,
        )

        next_sensor_reference_values.update(refreshed_refs)
        next_local_margin_values = refreshed_margins
        next_prior_xbar = computed_xbar
        next_prior_delta_global = computed_delta

        event_resync_performed = 1

        request_message_count = sensor_count
        response_message_count = sensor_count
        broadcast_message_count = sensor_count

        # Forced rows consume scheduled re-sync and must not emit local trigger requests.
        for sensor_col in sensor_cols:
            sensor_name = sensor_col.replace("_temperature", "")
            observed_sensor_col = f"observed_{sensor_name}_temperature"
            sensor_ref_col = f"entry_{sensor_name}_reference_value"
            local_dev_col = f"event_{sensor_name}_local_deviation"
            local_margin_col = f"entry_{sensor_name}_local_margin"
            margin_proximity_col = f"event_{sensor_name}_margin_proximity_score"
            trigger_col = f"event_{sensor_name}_resync_requested"

            sensor_val = row[sensor_col]
            ref_val = sensor_reference_values.get(sensor_col, np.nan)
            local_margin = local_margin_values.get(sensor_col, np.nan)

            sensor_fields[observed_sensor_col] = round_or_blank(sensor_val)
            sensor_fields[sensor_ref_col] = round_or_blank(ref_val)
            sensor_fields[local_margin_col] = round_or_blank(local_margin)
            sensor_fields[local_dev_col] = ""
            sensor_fields[margin_proximity_col] = ""
            sensor_fields[trigger_col] = 0

            if (
                not np.isnan(sensor_val)
                and not np.isnan(ref_val)
                and not np.isnan(local_margin)
                and local_margin < 0
                and np.isclose(float(sensor_val) - float(ref_val), 0.0)
            ):
                assert sensor_fields[trigger_col] == 0

        event_resync_request_count = 0
        event_any_sensor_requested_resync = 0
        sensor_last_resync_included = {
            sensor_col: sensor_col in refreshed_refs for sensor_col in sensor_cols
        }
    else:
        for sensor_col in sensor_cols:
            sensor_name = sensor_col.replace("_temperature", "")
            observed_sensor_col = f"observed_{sensor_name}_temperature"
            sensor_ref_col = f"entry_{sensor_name}_reference_value"
            local_dev_col = f"event_{sensor_name}_local_deviation"
            local_margin_col = f"entry_{sensor_name}_local_margin"
            margin_proximity_col = f"event_{sensor_name}_margin_proximity_score"
            trigger_col = f"event_{sensor_name}_resync_requested"

            sensor_val = row[sensor_col]
            ref_val = sensor_reference_values.get(sensor_col, np.nan)
            local_margin = local_margin_values.get(sensor_col, np.nan)

            sensor_fields[observed_sensor_col] = round_or_blank(sensor_val)
            sensor_fields[sensor_ref_col] = round_or_blank(ref_val)
            sensor_fields[local_margin_col] = round_or_blank(local_margin)

            if np.isnan(sensor_val) or np.isnan(ref_val):
                sensor_fields[local_dev_col] = ""
                sensor_fields[margin_proximity_col] = ""
                sensor_fields[trigger_col] = 0
                continue

            local_deviation = float(sensor_val) - float(ref_val)
            sensor_fields[local_dev_col] = round(local_deviation, 4)
            margin_proximity = float(entry_delta_global) - float(local_deviation)
            sensor_fields[margin_proximity_col] = round(margin_proximity, 4)

            if np.isnan(local_margin):
                trigger_bit = 0
            else:
                trigger_bit = int(local_deviation >= float(local_margin))

            sensor_fields[trigger_col] = trigger_bit
            if trigger_bit == 1:
                triggering_sensor_names.append(sensor_name)

        event_resync_request_count = int(len(triggering_sensor_names))
        event_any_sensor_requested_resync = int(event_resync_request_count > 0)

        if event_resync_request_count > 0:
            trigger_message_count = event_resync_request_count

            (
                computed_xbar,
                computed_delta,
                refreshed_refs,
                refreshed_margins,
            ) = compute_resync_update(
                row=row,
                sensor_columns=sensor_cols,
                global_average=global_avg,
                fallback_xbar=prior_xbar,
                threshold=p90_threshold,
            )

            next_sensor_reference_values.update(refreshed_refs)
            next_local_margin_values = refreshed_margins
            next_prior_xbar = computed_xbar
            next_prior_delta_global = computed_delta

            event_resync_performed = 1
            event_resync_triggered_by_local_violation = 1
            event_resync_reason = "local_constraint_violation"

            request_message_count = sensor_count
            response_message_count = sensor_count
            broadcast_message_count = sensor_count
            sensor_last_resync_included = {
                sensor_col: sensor_col in refreshed_refs for sensor_col in sensor_cols
            }

    global_violation = int(
        (not np.isnan(global_avg)) and (float(global_avg) >= float(p90_threshold))
    )

    event_exit_delta_negative = int(next_prior_delta_global < 0)
    if event_exit_delta_negative == 1:
        exit_resync_scheduled_next_row = 1

    if event_resync_consumed_from_prior_row == 1:
        assert trigger_message_count == 0
        assert event_resync_triggered_by_local_violation == 0
        assert event_any_sensor_requested_resync == 0
    if recovery_resync_row and event_resync_consumed_from_prior_row == 0:
        assert event_resync_performed == 1

    total_message_count = int(
        trigger_message_count
        + request_message_count
        + response_message_count
        + broadcast_message_count
    )

    transition_summary = build_transition_summary(
        event_resync_consumed_from_prior_row=event_resync_consumed_from_prior_row,
        event_any_sensor_requested_resync=event_any_sensor_requested_resync,
        event_resync_triggered_by_local_violation=event_resync_triggered_by_local_violation,
        event_exit_delta_negative=event_exit_delta_negative,
        exit_resync_scheduled_next_row=exit_resync_scheduled_next_row,
        event_resync_reason=event_resync_reason,
    )

    out = {
        "bucket_epoch": int(row["bucket_epoch"]),
        "bucket_time_utc": row["bucket_time_utc"],
        "transition_summary": transition_summary,
        "p90_threshold_used": round(float(p90_threshold), 4),
        "observed_global_average": round_or_blank(global_avg),
        "entry_xbar_t0": round(entry_xbar_t0, 4),
        "exit_xbar_t0_if_resync": "" if event_resync_performed == 0 else round(float(next_prior_xbar), 4),
        "entry_delta_global": round(entry_delta_global, 4),
        "exit_delta_global_if_resync": "" if event_resync_performed == 0 else round(float(next_prior_delta_global), 4),
        "global_violation": global_violation,
        "event_triggering_sensor_names": "|".join(triggering_sensor_names),
        "event_resync_request_count": event_resync_request_count,
        "event_any_sensor_requested_resync": event_any_sensor_requested_resync,
        "event_resync_consumed_from_prior_row": event_resync_consumed_from_prior_row,
        "event_resync_triggered_by_local_violation": event_resync_triggered_by_local_violation,
        "event_resync_performed": event_resync_performed,
        "event_resync_reason": event_resync_reason,
        "event_exit_delta_negative": event_exit_delta_negative,
        "trigger_message_count": int(trigger_message_count),
        "request_message_count": int(request_message_count),
        "response_message_count": int(response_message_count),
        "broadcast_message_count": int(broadcast_message_count),
        "total_message_count": int(total_message_count),
        "centralized_constraint_state": "VIOLATING" if global_violation == 1 else "FEASIBLE",
        "distributed_constraint_state": "VIOLATING" if event_resync_performed == 1 else "FEASIBLE",
        "distributed_alert": int(event_resync_performed),
        "false_safe": int(global_violation == 1 and event_resync_performed == 0),
        "false_alert": int(global_violation == 0 and event_resync_performed == 1),
        "violation_detection_delay_buckets": "",
    }

    out.update(sensor_fields)
    records.append(out)

    prior_xbar = float(next_prior_xbar)
    prior_delta_global = float(next_prior_delta_global)
    sensor_reference_values = dict(next_sensor_reference_values)
    local_margin_values = dict(next_local_margin_values)
    resync_due_next_row = int(exit_resync_scheduled_next_row)


phase2_df = pd.DataFrame.from_records(records)
public_phase2_df = phase2_df.drop(
    columns=[
        "event_resync_consumed_from_prior_row",
        "event_resync_triggered_by_local_violation",
    ],
    errors="ignore",
)
print(f"Wrote {len(phase2_df)} rows to {PHASE2_OUTPUT_PATH}")

PHASE2_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
public_phase2_df.to_csv(PHASE2_OUTPUT_PATH, index=False)




## Stage 3: Metrics and Communication Diagnostics

This stage derives classification-style diagnostics, communication cost comparisons, and detection delay summaries from the generated Phase 2 dataset.

Interpretation note: lag values are bucket offsets from the start of each contiguous violation window, not absolute row indices or timestamps.

In [ ]:
import re

In [ ]:
def compute_confusion_counts(actual_series, predicted_series):
    tp = int(((actual_series == 1) & (predicted_series == 1)).sum())
    fn = int(((actual_series == 1) & (predicted_series == 0)).sum())
    fp = int(((actual_series == 0) & (predicted_series == 1)).sum())
    tn = int(((actual_series == 0) & (predicted_series == 0)).sum())
    return tp, fn, fp, tn

def compute_window_lags(actual_series, predicted_series):
    # Lag unit is buckets since the start of each contiguous actual-positive window.
    lags = []
    in_window = False
    window_start = None

    for idx, val in enumerate(actual_series.tolist()):
        if val == 1 and not in_window:
            in_window = True
            window_start = idx
        elif val == 0 and in_window:
            window_end = idx - 1
            segment = predicted_series.iloc[window_start : window_end + 1]
            hit_positions = np.where(segment.to_numpy() == 1)[0]
            if len(hit_positions) > 0:
                lags.append(int(hit_positions[0]))
            else:
                lags.append(None)
            in_window = False

    if in_window and window_start is not None:
        segment = predicted_series.iloc[window_start:]
        hit_positions = np.where(segment.to_numpy() == 1)[0]
        if len(hit_positions) > 0:
            lags.append(int(hit_positions[0]))
        else:
            lags.append(None)

    return lags

def round4(value):
    return round(float(value), 4)

def compute_row_violation_detection_delay(actual_series, predicted_series):
    # For each violating row, emit the first-hit lag within its contiguous violation window.
    row_delay = [None] * len(actual_series)
    actual_list = actual_series.tolist()
    predicted_list = predicted_series.tolist()

    in_window = False
    window_start = None

    for idx, val in enumerate(actual_list):
        if val == 1 and not in_window:
            in_window = True
            window_start = idx
        elif val == 0 and in_window:
            window_end = idx - 1
            segment = predicted_list[window_start : window_end + 1]
            hit_positions = np.where(np.array(segment) == 1)[0]
            lag_value = int(hit_positions[0]) if len(hit_positions) > 0 else None
            for window_idx in range(window_start, window_end + 1):
                row_delay[window_idx] = lag_value
            in_window = False

    if in_window and window_start is not None:
        segment = predicted_list[window_start:]
        hit_positions = np.where(np.array(segment) == 1)[0]
        lag_value = int(hit_positions[0]) if len(hit_positions) > 0 else None
        for window_idx in range(window_start, len(actual_list)):
            row_delay[window_idx] = lag_value

    return row_delay

actual = phase2_df["global_violation"].astype(int)
predicted = phase2_df["event_resync_performed"].astype(int)

tp, fn, fp, tn = compute_confusion_counts(actual, predicted)

recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan

sensor_count = len(sensor_cols)
baseline_sensor_messages = len(phase2_df) * sensor_count

distributed_total_messages = int(phase2_df["total_message_count"].sum())
distributed_trigger_messages = int(phase2_df["trigger_message_count"].sum())
distributed_request_messages = int(phase2_df["request_message_count"].sum())
distributed_response_messages = int(phase2_df["response_message_count"].sum())
distributed_broadcast_messages = int(phase2_df["broadcast_message_count"].sum())

# Primary ratio uses total distributed traffic; legacy ratio preserves old trigger-only baseline comparison.
communication_reduction_ratio_primary = (
    1.0 - (distributed_total_messages / baseline_sensor_messages)
    if baseline_sensor_messages > 0
    else np.nan
)
communication_reduction_ratio_legacy = (
    1.0 - (distributed_trigger_messages / baseline_sensor_messages)
    if baseline_sensor_messages > 0
    else np.nan
)

window_lags = compute_window_lags(actual, predicted)
valid_lags = [x for x in window_lags if x is not None]
row_detection_delay = compute_row_violation_detection_delay(actual, predicted)
phase2_df["violation_detection_delay_buckets"] = row_detection_delay

# Ensure observability fields are present even if older artifacts predate these columns.
if "distributed_alert" not in phase2_df.columns:
    phase2_df["distributed_alert"] = phase2_df["event_resync_performed"].astype(int)
if "centralized_constraint_state" not in phase2_df.columns:
    phase2_df["centralized_constraint_state"] = np.where(
        phase2_df["global_violation"].astype(int) == 1,
        "VIOLATING",
        "FEASIBLE",
    )
if "distributed_constraint_state" not in phase2_df.columns:
    phase2_df["distributed_constraint_state"] = np.where(
        phase2_df["distributed_alert"].astype(int) == 1,
        "VIOLATING",
        "FEASIBLE",
    )
if "false_safe" not in phase2_df.columns:
    phase2_df["false_safe"] = (
        (phase2_df["centralized_constraint_state"] == "VIOLATING")
        & (phase2_df["event_resync_performed"].astype(int) == 0)
    ).astype(int)
if "false_alert" not in phase2_df.columns:
    phase2_df["false_alert"] = (
        (phase2_df["centralized_constraint_state"] == "FEASIBLE")
        & (phase2_df["event_resync_performed"].astype(int) == 1)
    ).astype(int)

false_safe_count = int(phase2_df["false_safe"].sum())
false_alert_count = int(phase2_df["false_alert"].sum())

# Metrics sections document rule definitions, quality diagnostics, and communication cost.
metrics = {
    "rows": int(len(phase2_df)),
    "sensor_count": int(sensor_count),
    "p90_threshold": round4(p90_threshold),
    "boundary_policy": {
        "canonical_rule": "violated_if_observed_global_average_gte_p90",
        "variant_policy": "strict_gt_available_for_diagnostic_sensitivity_only",
    },
    "trigger_policy": {
        "local_trigger_rule": "event_{sensor}_local_deviation >= entry_{sensor}_local_margin",
        "predicted_positive_rule": "event_resync_performed == 1",
    },
    "confusion": {
        "tp": tp,
        "fn": fn,
        "fp": fp,
        "tn": tn,
    },
    "false_negative_count": fn,
    "false_positive_count": fp,
    "false_safe_count": false_safe_count,
    "false_alert_count": false_alert_count,
    "recall_diagnostic": None if np.isnan(recall) else round4(recall),
    "precision_diagnostic": None if np.isnan(precision) else round4(precision),
    "communication": {
        "distributed_total_messages": distributed_total_messages,
        "distributed_trigger_messages": distributed_trigger_messages,
        "distributed_request_messages": distributed_request_messages,
        "distributed_response_messages": distributed_response_messages,
        "distributed_broadcast_messages": distributed_broadcast_messages,
        "centralized_sensor_messages": int(baseline_sensor_messages),
        "communication_reduction_ratio_primary": None
        if np.isnan(communication_reduction_ratio_primary)
        else round4(communication_reduction_ratio_primary),
        "communication_reduction_ratio_legacy": None
        if np.isnan(communication_reduction_ratio_legacy)
        else round4(communication_reduction_ratio_legacy),
    },
    "resync_events": {
        "resync_performed_count": int(phase2_df["event_resync_performed"].sum()),
        "negative_delta_detected_count": int(phase2_df["event_exit_delta_negative"].sum()),
        "scheduled_from_prior_count": int(phase2_df["event_resync_consumed_from_prior_row"].sum()),
    },
    "detection_delay_by_window": {
        "window_lags_in_buckets": window_lags,
        "mean_lag": None if not valid_lags else round4(np.mean(valid_lags)),
        "max_lag": None if not valid_lags else int(max(valid_lags)),
    },
}

with PHASE2_METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

# Rewrite row output so row-level detection delay and observability fields are persisted.
phase2_df.to_csv(PHASE2_OUTPUT_PATH, index=False)

PHASE2_DICTIONARY_PATH = REPO_ROOT / "artifacts/phase2_data_dictionary.csv"
PHASE2_ANALYSIS_PATH = REPO_ROOT / "artifacts/phase2_sensor_reduction_analysis.csv"

def _detect_sensors(columns):
    sensors = set()
    pattern = re.compile(r"^observed_(.+)_temperature$")
    for column in columns:
        match = pattern.match(column)
        if match:
            sensors.add(match.group(1))
    return sorted(sensors)

def _describe_column(column, sensor_names):
    static = {
        "bucket_epoch": {
            "category": "time",
            "measures": "Unix epoch second for the bucket row.",
            "calculation": "Copied from aligned Phase 1 bucket timestamp.",
            "units": "seconds since 1970-01-01 UTC",
        },
        "bucket_time_utc": {
            "category": "time",
            "measures": "UTC timestamp for the bucket row.",
            "calculation": "Formatted datetime representation of bucket_epoch.",
            "units": "UTC datetime",
        },
        "transition_summary": {
            "category": "state",
            "measures": "Compact state transition label for row auditability.",
            "calculation": "Composed from row event conditions in the state machine.",
            "units": "text",
        },
        "p90_threshold_used": {
            "category": "threshold",
            "measures": "Global P90 threshold used for this run.",
            "calculation": "Loaded from phase1_average_statistics.csv for the all-devices series all_devices_including_imputed.",
            "units": "temperature",
        },
        "observed_global_average": {
            "category": "global",
            "measures": "Current row global average temperature.",
            "calculation": "From Phase 1 aligned field average_temperature_all_devices using the strict all-devices policy.",
            "units": "temperature",
        },
        "entry_xbar_t0": {
            "category": "global",
            "measures": "Global reference average entering this row.",
            "calculation": "Prior synchronized global reference carried into this row.",
            "units": "temperature",
        },
        "entry_delta_global": {
            "category": "global",
            "measures": "Global margin entering this row.",
            "calculation": "p90_threshold_used - entry_xbar_t0.",
            "units": "temperature",
        },
        "exit_delta_global": {
            "category": "global",
            "measures": "Global margin after row event processing.",
            "calculation": "p90_threshold_used - exit global reference (post-event state).",
            "units": "temperature",
        },
        "global_violation": {
            "category": "diagnostic",
            "measures": "Whether row average violates canonical threshold boundary.",
            "calculation": "1 when observed_global_average >= p90_threshold_used, else 0.",
            "units": "binary (0/1)",
        },
        "event_triggering_sensor_names": {
            "category": "event",
            "measures": "Sensor names that requested re-sync on this row.",
            "calculation": "Pipe-delimited list of sensors with event_{sensor}_resync_requested == 1.",
            "units": "text",
        },
        "event_resync_request_count": {
            "category": "event",
            "measures": "Count of sensor-local re-sync requests on this row.",
            "calculation": "Sum of event_{sensor}_resync_requested over all sensors.",
            "units": "count",
        },
        "event_any_sensor_requested_resync": {
            "category": "event",
            "measures": "Whether any sensor requested re-sync on this row.",
            "calculation": "1 when event_resync_request_count > 0, else 0.",
            "units": "binary (0/1)",
        },
        "event_resync_performed": {
            "category": "event",
            "measures": "Whether a re-sync event was executed on this row.",
            "calculation": "1 when forced consumption or local trigger causes synchronization.",
            "units": "binary (0/1)",
        },
        "event_resync_reason": {
            "category": "event",
            "measures": "Reason code for performed re-sync.",
            "calculation": "Categorical label from row event evaluation.",
            "units": "text",
        },
        "event_exit_delta_negative": {
            "category": "event",
            "measures": "Whether the row exit global margin is negative.",
            "calculation": "1 when exit_delta_global < 0, else 0.",
            "units": "binary (0/1)",
        },
        "exit_resync_scheduled_next_row": {
            "category": "state",
            "measures": "Whether next row must consume a forced re-sync.",
            "calculation": "Set to 1 when current row exit_delta_global is negative.",
            "units": "binary (0/1)",
        },
        "exit_xbar_t0_if_resync": {
            "category": "state",
            "measures": "Global reference that becomes next row entry_xbar_t0 after re-sync.",
            "calculation": "Set to observed_global_average when re-sync is performed; otherwise carry prior.",
            "units": "temperature",
        },
        "exit_delta_global_if_resync": {
            "category": "state",
            "measures": "Global margin that becomes next row entry_delta_global after re-sync.",
            "calculation": "Set to p90_threshold_used - exit_xbar_t0_if_resync.",
            "units": "temperature",
        },
        "trigger_message_count": {
            "category": "communication",
            "measures": "Number of sensor trigger messages emitted this row.",
            "calculation": "On local-trigger rows equals event_resync_request_count; forced rows fixed to 0.",
            "units": "messages",
        },
        "request_message_count": {
            "category": "communication",
            "measures": "Number of distributed request messages this row.",
            "calculation": "Fanout count recorded when re-sync is performed.",
            "units": "messages",
        },
        "response_message_count": {
            "category": "communication",
            "measures": "Number of distributed response messages this row.",
            "calculation": "Fanout count recorded when re-sync is performed.",
            "units": "messages",
        },
        "broadcast_message_count": {
            "category": "communication",
            "measures": "Number of distributed broadcast messages this row.",
            "calculation": "Fanout count recorded when re-sync is performed.",
            "units": "messages",
        },
        "total_message_count": {
            "category": "communication",
            "measures": "Total communication messages for this row.",
            "calculation": "trigger_message_count + request_message_count + response_message_count + broadcast_message_count.",
            "units": "messages",
        },
        "centralized_constraint_state": {
            "category": "correctness",
            "measures": "Centralized oracle threshold state for this row.",
            "calculation": "VIOLATING when global_violation == 1, else FEASIBLE.",
            "units": "categorical",
        },
        "distributed_constraint_state": {
            "category": "correctness",
            "measures": "Distributed monitor state representation for this row.",
            "calculation": "VIOLATING when event_resync_performed == 1, else FEASIBLE.",
            "units": "categorical",
        },
        "distributed_alert": {
            "category": "correctness",
            "measures": "Distributed implemented-resync indicator for this row.",
            "calculation": "Mirrors event_resync_performed.",
            "units": "binary (0/1)",
        },
        "false_safe": {
            "category": "correctness",
            "measures": "Rows where centralized state violates and no distributed re-sync was implemented on that row.",
            "calculation": "1 when global_violation == 1 and event_resync_performed == 0.",
            "units": "binary (0/1)",
        },
        "false_alert": {
            "category": "correctness",
            "measures": "Rows where a distributed re-sync was implemented while centralized state is feasible.",
            "calculation": "1 when global_violation == 0 and event_resync_performed == 1.",
            "units": "binary (0/1)",
        },
        "violation_detection_delay_buckets": {
            "category": "correctness",
            "measures": "Per-row delay marker for first distributed detection within a contiguous violation window.",
            "calculation": "Window-level first-hit lag projected onto rows belonging to that violation window.",
            "units": "buckets",
        },
    }

    if column in static:
        return {
            "field_name": column,
            "category": static[column]["category"],
            "measures": static[column]["measures"],
            "calculation": static[column]["calculation"],
            "units": static[column]["units"],
        }

    sensor_observed = re.match(r"^observed_(.+)_temperature$", column)
    if sensor_observed:
        sensor = sensor_observed.group(1)
        if sensor in sensor_names:
            return {
                "field_name": column,
                "category": "sensor_observed",
                "measures": f"Observed temperature for sensor {sensor} on this row.",
                "calculation": "Direct carry-forward of aligned Phase 1 per-sensor value for the same bucket.",
                "units": "temperature",
            }

    sensor_reference = re.match(r"^entry_(.+)_reference_value$", column)
    if sensor_reference:
        sensor = sensor_reference.group(1)
        return {
            "field_name": column,
            "category": "sensor_state",
            "measures": f"Entry synchronized reference value for sensor {sensor}.",
            "calculation": "Sensor value stored at most recent synchronization event.",
            "units": "temperature",
        }

    sensor_margin = re.match(r"^entry_(.+)_local_margin$", column)
    if sensor_margin:
        sensor = sensor_margin.group(1)
        return {
            "field_name": column,
            "category": "sensor_state",
            "measures": f"Entry local trigger margin for sensor {sensor}.",
            "calculation": f"p90_threshold_used - entry_{sensor}_reference_value.",
            "units": "temperature",
        }

    sensor_deviation = re.match(r"^event_(.+)_local_deviation$", column)
    if sensor_deviation:
        sensor = sensor_deviation.group(1)
        return {
            "field_name": column,
            "category": "sensor_event",
            "measures": f"Row local deviation for sensor {sensor}.",
            "calculation": f"observed_{sensor}_temperature - entry_{sensor}_reference_value.",
            "units": "temperature",
        }

    sensor_margin_score = re.match(r"^event_(.+)_margin_proximity_score$", column)
    if sensor_margin_score:
        sensor = sensor_margin_score.group(1)
        return {
            "field_name": column,
            "category": "sensor_event",
            "measures": f"Row margin proximity score for sensor {sensor} (positive=acceptable, negative=violating).",
            "calculation": f"entry_delta_global - event_{sensor}_local_deviation.",
            "units": "temperature",
        }

    sensor_request = re.match(r"^event_(.+)_resync_requested$", column)
    if sensor_request:
        sensor = sensor_request.group(1)
        return {
            "field_name": column,
            "category": "sensor_event",
            "measures": f"Whether sensor {sensor} requested re-sync on this row.",
            "calculation": f"1 when event_{sensor}_local_deviation >= entry_{sensor}_local_margin and row is not forced; else 0.",
            "units": "binary (0/1)",
        }

    return {
        "field_name": column,
        "category": "other",
        "measures": "Field included in Phase 2 output schema.",
        "calculation": "See scripts/README.md and notebook logic for detailed derivation.",
        "units": "n/a",
    }

def _build_data_dictionary(raw_columns, sensor_names):
    sensor_set = set(sensor_names)
    rows = [_describe_column(column, sensor_set) for column in raw_columns]
    return pd.DataFrame(rows)

def _as_number(value, field_name):
    if isinstance(value, (int, float)):
        return float(value)
    raise ValueError(f"Expected numeric value for '{field_name}', received: {value!r}")

def _build_reduction_analysis(metrics_dict):
    communication = metrics_dict.get("communication")
    if not isinstance(communication, dict):
        raise ValueError("Missing 'communication' object in metrics JSON.")

    rows = _as_number(metrics_dict.get("rows"), "rows")
    sensor_count = _as_number(metrics_dict.get("sensor_count"), "sensor_count")
    baseline = rows * sensor_count

    actual = _as_number(
        communication.get("distributed_total_messages"), "distributed_total_messages"
    )
    trigger_messages = _as_number(
        communication.get("distributed_trigger_messages"), "distributed_trigger_messages"
    )
    request_messages = _as_number(
        communication.get("distributed_request_messages"), "distributed_request_messages"
    )
    response_messages = _as_number(
        communication.get("distributed_response_messages"), "distributed_response_messages"
    )
    broadcast_messages = _as_number(
        communication.get("distributed_broadcast_messages"), "distributed_broadcast_messages"
    )

    if baseline <= 0:
        raise ValueError("Baseline communications must be > 0.")

    difference = baseline - actual
    ratio = 1.0 - (actual / baseline)
    percent = round(ratio * 100.0, 2)

    false_safe_count = metrics_dict.get("false_safe_count")
    false_alert_count = metrics_dict.get("false_alert_count")

    analysis_rows = [
        {
            "metric": "baseline_every_minute_messages",
            "value": int(round(baseline)),
            "units": "messages",
            "calculation": "rows * sensor_count",
            "notes": "Counterfactual baseline if every sensor transmitted every minute.",
        },
        {
            "metric": "poc_actual_messages",
            "value": int(round(actual)),
            "units": "messages",
            "calculation": "distributed_total_messages",
            "notes": "Actual total distributed traffic under this Phase 2 implementation.",
        },
        {
            "metric": "reduction_difference_messages",
            "value": int(round(difference)),
            "units": "messages",
            "calculation": "baseline_every_minute_messages - poc_actual_messages",
            "notes": "Absolute communication reduction achieved by the POC.",
        },
        {
            "metric": "reduction_percent_primary",
            "value": f"{percent:.2f}",
            "units": "percent",
            "calculation": "reduction_ratio_primary * 100",
            "notes": "Primary reduction expressed as percentage.",
        },
        {
            "metric": "distributed_trigger_messages",
            "value": int(round(trigger_messages)),
            "units": "messages",
            "calculation": "sum(trigger_message_count)",
            "notes": "Trigger-category communication breakdown.",
        },
        {
            "metric": "distributed_request_messages",
            "value": int(round(request_messages)),
            "units": "messages",
            "calculation": "sum(request_message_count)",
            "notes": "Request-category communication breakdown.",
        },
        {
            "metric": "distributed_response_messages",
            "value": int(round(response_messages)),
            "units": "messages",
            "calculation": "sum(response_message_count)",
            "notes": "Response-category communication breakdown.",
        },
        {
            "metric": "distributed_broadcast_messages",
            "value": int(round(broadcast_messages)),
            "units": "messages",
            "calculation": "sum(broadcast_message_count)",
            "notes": "Broadcast-category communication breakdown.",
        },
        {
            "metric": "rows",
            "value": int(round(rows)),
            "units": "count",
            "calculation": "metrics.rows",
            "notes": "Number of Phase 2 minute buckets.",
        },
        {
            "metric": "sensor_count",
            "value": int(round(sensor_count)),
            "units": "count",
            "calculation": "metrics.sensor_count",
            "notes": "Number of sensors included in Phase 2 run.",
        },
    ]

    resync = metrics_dict.get("resync_events")
    if isinstance(resync, dict) and isinstance(
        resync.get("resync_performed_count"), (int, float)
    ):
        analysis_rows.append(
            {
                "metric": "resync_performed_count",
                "value": int(round(float(resync["resync_performed_count"]))),
                "units": "count",
                "calculation": "sum(event_resync_performed)",
                "notes": "Count of synchronization events in the run.",
            }
        )

    if isinstance(false_safe_count, (int, float)):
        analysis_rows.append(
            {
                "metric": "false_safe_count",
                "value": int(round(float(false_safe_count))),
                "units": "count",
                "calculation": "sum(false_safe)",
                "notes": "Row-level observability counter for violating rows with no implemented distributed re-sync.",
            }
        )

    if isinstance(false_alert_count, (int, float)):
        analysis_rows.append(
            {
                "metric": "false_alert_count",
                "value": int(round(float(false_alert_count))),
                "units": "count",
                "calculation": "sum(false_alert)",
                "notes": "Row-level observability counter for implemented distributed re-sync rows on centralized FEASIBLE minutes.",
            }
        )

    reduction_analysis = pd.DataFrame(analysis_rows)
    if "bucket_epoch" in reduction_analysis.columns:
        reduction_analysis = reduction_analysis.drop(columns=["bucket_epoch"])
    return reduction_analysis

def _write_csv_atomic(df, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = output_path.with_suffix(output_path.suffix + ".tmp")
    rounded_df = df.copy()
    for column in rounded_df.columns:
        if rounded_df[column].dtype.kind in "ifc":
            rounded_df[column] = rounded_df[column].round(4)
    rounded_df.to_csv(tmp_path, index=False)
    try:
        tmp_path.replace(output_path)
    except PermissionError:
        rounded_df.to_csv(output_path, index=False)
        if tmp_path.exists():
            tmp_path.unlink()

def _export_phase2_report_csvs_from_artifacts(rows_input, metrics_input, data_dictionary_output, analysis_output):
    if not rows_input.exists():
        raise FileNotFoundError(f"Missing rows input CSV: {rows_input}")
    if not metrics_input.exists():
        raise FileNotFoundError(f"Missing metrics input JSON: {metrics_input}")

    raw_rows = pd.read_csv(rows_input)
    metrics_dict = json.loads(metrics_input.read_text(encoding="utf-8"))

    sensor_names = _detect_sensors(list(raw_rows.columns))
    data_dictionary = _build_data_dictionary(list(raw_rows.columns), sensor_names)
    reduction_analysis = _build_reduction_analysis(metrics_dict)

    _write_csv_atomic(data_dictionary, data_dictionary_output)
    _write_csv_atomic(reduction_analysis, analysis_output)

    return {
        "data_dictionary": data_dictionary_output,
        "sensor_reduction_analysis": analysis_output,
    }

report_outputs = _export_phase2_report_csvs_from_artifacts(
    rows_input=PHASE2_OUTPUT_PATH,
    metrics_input=PHASE2_METRICS_PATH,
    data_dictionary_output=PHASE2_DICTIONARY_PATH,
    analysis_output=PHASE2_ANALYSIS_PATH,
)

print(json.dumps(metrics, indent=2))
print(f"Wrote metrics to {PHASE2_METRICS_PATH}")
print("Wrote CSV report outputs:")
for key, value in report_outputs.items():
    print(f"- {key}: {value}")

## Stage 4: Integrated Validation Checks

This stage consolidates smoke checks and export-policy checks directly into notebook execution.

It validates:
- required input/output artifact presence,
- required Phase 2 schema families and column ordering,
- numeric rounding policy,
- required metrics policy sections,
- required report artifacts and report-schema invariants.

Both report artifacts are required in this integrated flow:
- `artifacts/phase2_data_dictionary.csv`
- `artifacts/phase2_sensor_reduction_analysis.csv`

In [ ]:
import re


def _has_more_than_four_decimals(value):
    if pd.isna(value):
        return False
    try:
        numeric_value = float(value)
    except (TypeError, ValueError):
        return False
    return abs(numeric_value - round(numeric_value, 4)) > 1e-12


def _check_phase2_schema(rows_df, analysis_df):
    errors = []

    if "exit_delta_global" in rows_df.columns:
        errors.append("phase2_temperature_sensors.csv still includes exit_delta_global")

    columns = list(rows_df.columns)
    required_columns = [
        "entry_xbar_t0",
        "exit_xbar_t0_if_resync",
        "entry_delta_global",
        "exit_delta_global_if_resync",
    ]
    for column in required_columns:
        if column not in columns:
            errors.append(f"Missing expected schema column: {column}")

    if "entry_xbar_t0" in columns and "exit_xbar_t0_if_resync" in columns:
        entry_index = columns.index("entry_xbar_t0")
        exit_index = columns.index("exit_xbar_t0_if_resync")
        if exit_index != entry_index + 1:
            errors.append("exit_xbar_t0_if_resync is not immediately after entry_xbar_t0")

    if "entry_delta_global" in columns and "exit_delta_global_if_resync" in columns:
        entry_index = columns.index("entry_delta_global")
        exit_index = columns.index("exit_delta_global_if_resync")
        if exit_index != entry_index + 1:
            errors.append("exit_delta_global_if_resync is not immediately after entry_delta_global")

    local_margin_columns = [
        column for column in columns if re.match(r"^entry_.+_local_margin$", column)
    ]
    margin_score_columns = [
        column for column in columns if re.match(r"^event_.+_margin_proximity_score$", column)
    ]

    print("phase2_local_margin_cols", len(local_margin_columns))
    print("phase2_margin_proximity_cols", len(margin_score_columns))

    if not local_margin_columns:
        errors.append("Missing expected local margin columns in phase2_temperature_sensors.csv")
    if not margin_score_columns:
        errors.append("Missing expected margin proximity score columns in phase2_temperature_sensors.csv")

    if "bucket_epoch" in analysis_df.columns:
        errors.append("phase2_sensor_reduction_analysis.csv still includes bucket_epoch")

    return errors


def _check_phase2_rounding(rows_df):
    errors = []
    tokens = ["temperature", "average", "threshold", "delta", "xbar", "margin", "deviation", "reference"]

    for column in rows_df.columns:
        lowered = column.lower()
        if any(token in lowered for token in tokens):
            values = rows_df[column].dropna()
            if any(_has_more_than_four_decimals(value) for value in values):
                errors.append(f"{column} contains values with more than 4 decimal places")

    return errors


def _check_phase2_metrics(metrics_dict):
    errors = []
    for key in ["boundary_policy", "trigger_policy"]:
        if key not in metrics_dict:
            errors.append(f"Missing metrics field: {key}")
    return errors


def _check_phase2_dictionary(dictionary_df, rows_df):
    errors = []
    expected_columns = ["field_name", "category", "measures", "calculation", "units"]
    missing_columns = [column for column in expected_columns if column not in dictionary_df.columns]
    if missing_columns:
        errors.append(f"phase2_data_dictionary.csv missing columns: {missing_columns}")

    if len(dictionary_df) != len(rows_df.columns):
        errors.append(
            "phase2_data_dictionary.csv row count does not match phase2_temperature_sensors.csv column count"
        )

    if "field_name" in dictionary_df.columns:
        dictionary_fields = set(dictionary_df["field_name"].astype(str))
        row_fields = set(rows_df.columns)
        missing_from_dictionary = sorted(row_fields - dictionary_fields)
        if missing_from_dictionary:
            errors.append(
                "phase2_data_dictionary.csv missing field definitions: "
                + ", ".join(missing_from_dictionary[:10])
            )

    return errors


def _check_reduction_analysis_minimum_metrics(analysis_df):
    errors = []
    required_metrics = {
        "baseline_every_minute_messages",
        "poc_actual_messages",
        "reduction_difference_messages",
        "reduction_percent_primary",
    }

    if "metric" not in analysis_df.columns:
        errors.append("phase2_sensor_reduction_analysis.csv missing metric column")
        return errors

    observed = set(analysis_df["metric"].astype(str))
    missing = sorted(required_metrics - observed)
    if missing:
        errors.append(
            "phase2_sensor_reduction_analysis.csv missing required metrics: "
            + ", ".join(missing)
        )

    return errors


aligned_exists = PHASE1_ALIGNED_PATH.exists()
stats_exists = PHASE1_STATS_PATH.exists()
phase2_exists = PHASE2_OUTPUT_PATH.exists()
metrics_exists = PHASE2_METRICS_PATH.exists()
dictionary_exists = PHASE2_DICTIONARY_PATH.exists()
analysis_exists = PHASE2_ANALYSIS_PATH.exists()

print("aligned_exists", aligned_exists)
print("stats_exists", stats_exists)
print("phase2_exists", phase2_exists)
print("metrics_exists", metrics_exists)
print("dictionary_exists", dictionary_exists)
print("analysis_exists", analysis_exists)

if not aligned_exists:
    raise FileNotFoundError(f"Missing artifact: {PHASE1_ALIGNED_PATH}")
if not stats_exists:
    raise FileNotFoundError(f"Missing artifact: {PHASE1_STATS_PATH}")
if not phase2_exists:
    raise FileNotFoundError(f"Missing artifact: {PHASE2_OUTPUT_PATH}")
if not metrics_exists:
    raise FileNotFoundError(f"Missing artifact: {PHASE2_METRICS_PATH}")
if not dictionary_exists:
    raise FileNotFoundError(f"Missing artifact: {PHASE2_DICTIONARY_PATH}")
if not analysis_exists:
    raise FileNotFoundError(f"Missing artifact: {PHASE2_ANALYSIS_PATH}")

aligned_sample = pd.read_csv(PHASE1_ALIGNED_PATH, nrows=5)
sensor_columns = [
    column
    for column in aligned_sample.columns
    if column.endswith("_temperature")
    and column != "average_temperature_all_devices"
    and column != "average_temperature_non_imputed_devices"
]
print("sample_rows", len(aligned_sample))
print("sensor_cols", len(sensor_columns))

rows_df = pd.read_csv(PHASE2_OUTPUT_PATH)
metrics_on_disk = json.loads(PHASE2_METRICS_PATH.read_text(encoding="utf-8"))
dictionary_df = pd.read_csv(PHASE2_DICTIONARY_PATH)
analysis_df = pd.read_csv(PHASE2_ANALYSIS_PATH)

validation_errors = []
validation_errors.extend(_check_phase2_schema(rows_df, analysis_df))
validation_errors.extend(_check_phase2_rounding(rows_df))
validation_errors.extend(_check_phase2_metrics(metrics_on_disk))
validation_errors.extend(_check_phase2_dictionary(dictionary_df, rows_df))
validation_errors.extend(_check_reduction_analysis_minimum_metrics(analysis_df))

if validation_errors:
    print("Phase 2 notebook-integrated validation failed:")
    for error in validation_errors:
        print(f"- {error}")
    raise ValueError("Notebook-integrated Phase 2 validation failed")

print("Phase 2 notebook-integrated validation passed.")